# 05 - Baseline B4: HistGradientBoostingClassifier

B4 ya fue entrenado y sus modelos y resultados están en `results/baseline4/`. Las secciones siguientes conservan el procedimiento de búsqueda y ajuste final que produjo esos resultados, además de LightGBM, ablaciones y H2.

**Los interruptores de entrenamiento están en `False` por defecto.** Ejecutar las celdas con ese valor solo prepara el entorno y consulta los resultados existentes; las celdas informan que no entrenaron. Para una reproducción nueva, cambia los interruptores indicados a `True`: puede tardar horas y crea una carpeta nueva en Drive. La copia exacta del cuaderno que se usó originalmente está en `docx/03Evidence/05_entrenamiento_B4_independiente_original.ipynb`.


In [ ]:
# Cada cuaderno prepara su propio runtime de Colab; no depende de 00_arranque ni de otra sesión.
from pathlib import Path
import sys, json, time, subprocess
from datetime import datetime, timezone
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo fetch origin 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo checkout 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo pull --ff-only origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / 'src') not in sys.path: sys.path.insert(0, str(REPO_DIR / 'src'))
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp.ingest import load_days
from ptcg_wp.features import feature_columns
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", f"Se requiere scikit-learn 1.7.2; versión cargada: {sklearn_version}. Reinicia el runtime y ejecuta la preparación de nuevo."
from ptcg_wp.b4 import materialize_split, fit_b4_candidates_val
from ptcg_wp.pending_experiments import run_b4_ablation, run_h2, run_lightgbm
DATA_DIR = Path('/content/datos')
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(p.exists() for p in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(p.exists() for p in esperados), 'Faltan tablas procesadas; revisa Kaggle.'
schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62
SEED = 2026
SCENARIOS = ('A', 'B', 'C')
DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
RUN_INFO = {'commit': COMMIT, 'working_tree_dirty': DIRTY, 'seed': SEED, 'run_id_utc': RUN_ID}
RESULTS_DIR = REPO_DIR / 'results' / 'baseline4'
print('Tablas listas:', len(list(DATA_DIR.glob('estados_*.parquet'))), 'días | commit:', COMMIT,
      '| cambios locales:', DIRTY, '| corrida UTC:', RUN_ID)


## B4 original: búsqueda y entrenamiento final

Estas celdas muestran la configuración y las llamadas utilizadas para entrenar B4. El cuaderno histórico recuperado conserva sus celdas originales sin modificar. Los modelos existentes se cargan desde el repositorio si los interruptores se mantienen en `False`. Para una reproducción con entrenamiento, usa CPU + High-RAM y activa cada sección de forma explícita.

### Parte 1: búsqueda de hiperparámetros

Una búsqueda TPE, 20 trials y tres pliegues agrupados por `game_id`, sobre las 11 285 partidas que son `train` en A, B y C. Optimiza log loss y registra AUC y Brier. El ajuste de cada pliegue usa vigilancia agrupada, paciencia 30 y máximo 2000 iteraciones. El trial 20 fue el finalista ya versionado. La búsqueda original tardó aproximadamente 2 h 20 min con ocho núcleos.

In [ ]:
# Diagnóstico antes de una reproducción costosa de B4.
import os
try:
    import psutil
    ram_gib = psutil.virtual_memory().total / 1024**3
    print('CPU:', os.cpu_count(), '| RAM total GiB:', round(ram_gib, 1))
except ImportError:
    print('CPU:', os.cpu_count(), '| RAM: psutil no está instalado')
print('Para reproducir el entrenamiento original se recomienda CPU con 8 núcleos y High-RAM.')


In [ ]:
# Búsqueda original de B4. False consulta el CSV versionado; True reproduce la búsqueda.
EJECUTAR_BUSQUEDA_B4 = False
CHECKPOINT_ROOT = DRIVE_ROOT / 'b4_independiente'
RUN_ROOT = DRIVE_ROOT / 'b4_reproduccion' / RUN_ID / 'busqueda_v3'
features = FEATURES
B4_PATIENCE = 30
MAX_RAM_GIB = 10.0
SEED = 2026
if EJECUTAR_BUSQUEDA_B4:
    # B4 PARTE 1 - BUSQUEDA UNICA DE HIPERPARAMETROS
    # PROPOSITO: buscar hiperparametros UNA sola vez (Optuna TPE, validacion cruzada de 3 pliegues
    # agrupada por partida) usando SOLO partidas que son train en A, B y C a la vez. Asi ningun
    # hiperparametro vio partidas de val, calib o test de ningun escenario.
    # - Un solo estudio optimiza log loss; AUC y Brier de cada trial quedan registrados y definen
    #   las finalistas sin ajustes extra.
    # - Tope de 2000 arboles con parada temprana sobre un grupo de vigilancia separado.
    # - Todo se guarda en Drive por pliegue: si Colab se corta, volver a ejecutar reanuda.
    # - NO cambiar estos parametros: estan fijados en el manifiesto del estudio guardado en Drive.
    # ESPERADO: 11 285 partidas de busqueda. Si b4_candidatas.csv ya existe, la busqueda se omite
    # y solo se imprimen las finalistas (trial 20: AUC 0.838, log loss 0.488, Brier 0.163 en CV).
    from tempfile import TemporaryDirectory
    import gc
    import faulthandler
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    from ptcg_wp.baselines import ScenarioStream
    from ptcg_wp.b4 import materialize_split, b4_optuna_cv
    from ptcg_wp.checkpoints import atomic_csv
    from ptcg_wp.features import feature_columns

    features = feature_columns(pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names)
    B4_TRIALS = 20
    B4_SPLITS = 3
    B4_SEARCH_FRACTION = 1.0
    B4_MAX_ITER = 2000
    B4_PATIENCE = 30
    B4_OBJECTIVE = 'log_loss'
    MAX_RAM_GIB = 10.0
    SEED = 2026
    RUN_ROOT = DRIVE_ROOT / 'b4_reproduccion' / RUN_ID / 'busqueda_v3'
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    (RUN_ROOT / 'registro_reproduccion.json').write_text(json.dumps(RUN_INFO, indent=2))
    CANDIDATES_PATH = RUN_ROOT / 'busqueda' / 'b4_candidatas.csv'

    # Partidas que son train en los tres escenarios: las unicas que ve la busqueda
    part = pd.read_parquet(DATA_DIR / 'particiones.parquet')
    comunes = part.loc[(part['split_A'] == 'train') & (part['split_B'] == 'train')
                       & (part['split_C'] == 'train'), 'game_id']
    print('caracteristicas:', len(features), '| partidas de busqueda (train en A, B y C):', len(comunes))

    if not CANDIDATES_PATH.exists():
        trace_file = open('/content/b4_pipeline_tracebacks.log', 'a', buffering=1)
        faulthandler.enable(file=trace_file)
        faulthandler.dump_traceback_later(600, repeat=True, file=trace_file)
        try:
            print('B4: busqueda de hiperparametros', flush=True)
            with TemporaryDirectory(prefix='b4-busqueda-', dir='/content') as cache:
                stream = ScenarioStream(DATA_DIR, 'A', tuple(features), batch_size=32768)
                train = materialize_split(stream, features, 'train', cache)
                try:
                    codigos = np.flatnonzero(train.games.isin(comunes))
                    _, candidates = b4_optuna_cv(
                        train, n_splits=B4_SPLITS, n_trials=B4_TRIALS, timeout=None, seed=SEED,
                        max_iter=B4_MAX_ITER, patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                        output_dir=RUN_ROOT / 'busqueda',
                        study_cache_dir=str(Path('/content/ptcg_optuna') / f'b4_busqueda_{RUN_ID}'),
                        verbose=True, objective_metric=B4_OBJECTIVE,
                        search_fraction=B4_SEARCH_FRACTION, search_groups=codigos,
                    )
                    atomic_csv(candidates.to_dict('records'), CANDIDATES_PATH)
                finally:
                    train.close()
                del stream
            gc.collect()
        finally:
            faulthandler.cancel_dump_traceback_later()
            faulthandler.disable()
            trace_file.close()

    # Se leen siempre del CSV para que todas las sesiones usen exactamente las mismas finalistas
    candidates = pd.read_csv(CANDIDATES_PATH)
    print(candidates.to_string(index=False), flush=True)
else:
    CANDIDATES_PATH = REPO_DIR / 'results/baseline4/busqueda/b4_candidatas.csv'
    candidates = pd.read_csv(CANDIDATES_PATH)
    print('Búsqueda B4 NO ejecutada; se cargó la finalista ya versionada:', CANDIDATES_PATH)
    print(candidates.to_string(index=False))


### Parte 2: ajuste final por escenario

La finalista se ajusta en el `train` de cada escenario. Primero, el 15 % de las partidas de `train` sirve para decidir el número de árboles con parada temprana; después se reajusta con todo `train`, con tope de vigilancia de 4000. Los modelos versionados usaron 2888, 2370 y 2714 iteraciones para A, B y C. La ejecución original tardó aproximadamente 2 h 40 min con ocho núcleos.

In [ ]:
# Ajuste original de B4: False consulta artefactos existentes; True realiza una nueva reproducción.
EJECUTAR_AJUSTE_FINAL_B4 = False
if EJECUTAR_AJUSTE_FINAL_B4:
    from tempfile import TemporaryDirectory
    import gc
    from ptcg_wp.baselines import ScenarioStream
    from ptcg_wp.b4 import materialize_split
    from ptcg_wp.checkpoints import atomic_csv
    # B4 PARTE 2 - REENTRENAMIENTO FINAL CON TODO EL TRAIN DE CADA ESCENARIO (TOPE 4000)
    # PROPOSITO: reentrenar la finalista de la busqueda con TODO el train de cada escenario y
    # medirla en su val. La parada temprana (15 % de partidas de train como vigilancia) decide el
    # numero de arboles, con un tope de 4000 que no se alcanza (convergio entre 2370 y 2888).
    # - Requiere haber ejecutado la celda de la parte 1 (define features, candidates y parametros).
    # - Si un escenario ya termino, se omite (reanuda desde lo guardado en Drive).
    # - test NO se toca.
    # ESPERADO (medido): A arboles=2888 AUC=0.8346 | B arboles=2370 AUC=0.7863 |
    # C arboles=2714 AUC=0.8355. Modelos en busqueda_v3/final_4000/escenario_X/*.joblib.
    from ptcg_wp.b4 import fit_b4_candidates_val

    B4_MAX_ITER_FINAL = 4000
    FINAL_ROOT = RUN_ROOT / 'final_4000'

    for scenario in ('A', 'B', 'C'):
        output_dir = FINAL_ROOT / f'escenario_{scenario}'
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f'B4 {scenario}: preparando train y val', flush=True)
        with TemporaryDirectory(prefix=f'b4-final-{scenario}-', dir='/content') as cache:
            stream = ScenarioStream(DATA_DIR, scenario, tuple(features), batch_size=32768)
            train = materialize_split(stream, features, 'train', cache)
            val = materialize_split(stream, features, 'val', cache)
            try:
                val_metrics = fit_b4_candidates_val(
                    train, val, candidates, seed=SEED, max_iter=B4_MAX_ITER_FINAL,
                    patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=output_dir, scenario=scenario, verbose=True,
                )
                atomic_csv(val_metrics.to_dict('records'), output_dir / 'b4_val_finalistas.csv')
                g = val_metrics[val_metrics['etapa'] == 'global'].iloc[0]
                print(f"B4 {scenario}: terminado | arboles={g['n_iter_vigilancia']} | "
                      f"AUC={g['auc']:.4f} log_loss={g['log_loss']:.4f} brier={g['brier']:.4f}", flush=True)
            finally:
                train.close()
                val.close()
            del stream
        gc.collect()
    print('B4 listo. Modelos finales en:', FINAL_ROOT)
    (FINAL_ROOT / 'registro_reproduccion.json').write_text(json.dumps(RUN_INFO, indent=2))
else:
    FINAL_ROOT = REPO_DIR / 'results/baseline4/final_4000'
    print('Ajuste final B4 NO ejecutado; se usan los modelos ya versionados en', FINAL_ROOT)


## Métricas globales de validación

Se muestran por escenario; no se promedian. `test` permanece reservado para la etapa final.

In [ ]:
for escenario in ('A', 'B', 'C'):
    ruta = RESULTS_DIR / 'final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv'
    print(f'Escenario {escenario}')
    if ruta.exists():
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))
    else:
        print('No está disponible:', ruta)


## Configuración elegida

HistGradientBoostingClassifier, trial 20; `learning_rate=0.06943`, `max_leaf_nodes=31`, `min_samples_leaf=50` y `l2_regularization=4.2503`. Cada escenario se ajustó con su propio `train` y se evaluó en su `val`. Los modelos serializados están en `final_4000/escenario_X/`.

## Comparación con LightGBM: protocolo definido

Presupuesto comparable con B4: TPE con 20 trials y 3 pliegues por partida, sobre la intersección de partidas `train` de A/B/C. Optimiza log loss y registra AUC y Brier. Vigilancia agrupada del 15 %, paciencia 30, tope 2000 en búsqueda y 4000 en el ajuste final; semilla 2026. Una sola finalista se elige por mayor AUC de CV; los empates se resuelven por menor log loss, menor Brier y número de trial. Se elige antes de consultar `val` y se ajusta con el `train` propio de cada escenario. Son 60 ajustes de búsqueda y 6 para los modelos finales (vigilancia y reajuste en cada escenario).

Se seleccionará LightGBM solo si, en cada escenario A/B/C, el intervalo percentil del 95 % de la diferencia de AUC (LightGBM − B4) queda por encima de cero y no aumentan los valores observados de log loss ni Brier. Se usan 1000 remuestreos emparejados de partidas; cada partida conserva el mismo peso total. Si no se cumple toda la regla se mantiene B4 y se reportan las diferencias por escenario. Esta regla sirve para selección en `val`; no demuestra importancia práctica ni sustituye la evaluación final de etapa 05.


In [ ]:
# Ejecutar esta instalación antes de activar LightGBM en este runtime.
%pip install -q lightgbm==4.6.0


In [ ]:
from ptcg_wp.pending_experiments import run_lightgbm, compare_lightgbm_b4
EJECUTAR_LIGHTGBM = False
if EJECUTAR_LIGHTGBM:
    import lightgbm
    assert lightgbm.__version__ == '4.6.0', 'Instala la versión fijada en la celda anterior.'
    out = DRIVE_ROOT / 'lightgbm' / RUN_ID
    out.mkdir(parents=True, exist_ok=True)
    (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
    inicio = time.perf_counter()
    run_lightgbm(DATA_DIR, FEATURES, out, seed=SEED)
    segundos_ajuste = time.perf_counter()-inicio
    comparacion, decision = compare_lightgbm_b4(DATA_DIR,
        REPO_DIR / 'results/baseline4/final_4000', out, seed=SEED)
    (out / 'duracion.json').write_text(json.dumps({'segundos_busqueda_y_ajuste': segundos_ajuste,
        'segundos_comparacion': time.perf_counter()-inicio-segundos_ajuste,
        'segundos_total': time.perf_counter()-inicio}, indent=2))
    display(comparacion)
    print('Decisión sobre val:', decision['decision'])
    print('Guardado:', out)

else:
    print('LightGBM NO ejecutado: cambia EJECUTAR_LIGHTGBM a True para correr esta sección.')


## Ablaciones por familia

Tres variantes omiten una familia: progreso, recursos/tablero o diferenciales. La configuración de B4 se mantiene fija; cada variante ajusta con el `train` propio y se mide en `val` de cada escenario. Son nueve ajustes finales, además de sus ajustes de vigilancia. `familias.json` registra las columnas exactas.

In [ ]:
from ptcg_wp.pending_experiments import run_b4_ablation
EJECUTAR_ABLACIONES = False
ESCENARIOS_ABLACIONES = ('A', 'B', 'C')
if EJECUTAR_ABLACIONES:
    candidate = pd.read_csv(REPO_DIR / 'results/baseline4/busqueda/b4_candidatas.csv')
    candidate = candidate.loc[candidate['trial'].eq(20)].copy()
    assert len(candidate) == 1
    for escenario in ESCENARIOS_ABLACIONES:
        out = DRIVE_ROOT / 'b4' / 'ablaciones' / RUN_ID / f'escenario_{escenario}'
        out.mkdir(parents=True, exist_ok=True)
        (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
        run_b4_ablation(DATA_DIR, escenario, FEATURES, candidate, out, seed=SEED)
        print('Guardado:', out)

else:
    print('Ablaciones B4 NO ejecutado: cambia EJECUTAR_ABLACIONES a True para correr esta sección.')


## H2: estado completo frente a cuatro entradas

Reutiliza las métricas y el modelo B4 completo ya guardados. Ajusta la variante restringida con turno, primer jugador y los dos IDs activos. Los IDs se codifican como categorías con vocabulario de `train`; IDs no vistos en `val` se tratan como ausentes. Si hay más de 255 categorías, la celda se detiene para definir otra codificación. No se usa la variante de 64 entradas.

In [ ]:
from ptcg_wp.pending_experiments import run_h2
EJECUTAR_H2 = False
ESCENARIOS_H2 = ('A', 'B', 'C')
if EJECUTAR_H2:
    for escenario in ESCENARIOS_H2:
        out = DRIVE_ROOT / 'b4' / 'h2' / RUN_ID / f'escenario_{escenario}'
        out.mkdir(parents=True, exist_ok=True)
        (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
        run_h2(DATA_DIR, escenario, out, seed=SEED)
        completo = pd.read_csv(REPO_DIR / 'results/baseline4/final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv')
        restringido = pd.read_csv(out / 'h2_val.csv')
        print('B4 completo:', escenario); display(completo)
        print('H2 restringido:', escenario); display(restringido)
        print('Guardado:', out)

else:
    print('H2 NO ejecutado: cambia EJECUTAR_H2 a True para correr esta sección.')


## Resumen de la corrida actual

Muestra B4 completo y los resultados de ablaciones o LightGBM encontrados para el `RUN_ID` actual.

In [ ]:
# Después de ejecutar ablaciones o LightGBM, muestra sus métricas globales.
for escenario in ('A', 'B', 'C'):
    print('Escenario', escenario, '— B4 completo')
    full = pd.read_csv(REPO_DIR / 'results/baseline4/final_4000' / f'escenario_{escenario}' / 'b4_val_finalistas.csv')
    display(full.loc[full['etapa'].eq('global')].round(4))
    base = DRIVE_ROOT / 'b4' / 'ablaciones' / RUN_ID / f'escenario_{escenario}'
    for ruta in sorted(base.glob('sin_*/b4_val_finalists.csv')):
        print(ruta.parent.name)
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))
    ruta = DRIVE_ROOT / 'lightgbm' / RUN_ID / f'escenario_{escenario}' / 'lightgbm_val.csv'
    if ruta.exists():
        print('LightGBM — configuración finalista seleccionada en CV')
        datos = pd.read_csv(ruta)
        display(datos.loc[datos['etapa'].eq('global')].round(4))


## Descargar resultados B4

Empaqueta las salidas de reproducción de B4, ablaciones, H2 y LightGBM que existan bajo el `RUN_ID` actual, incluidos modelos y registros. Ejecuta esta celda después de terminar las secciones que quieras conservar.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import shutil

def agregar_a_zip(archivo_zip, ruta, raiz_drive, carpeta_zip):
    ruta = Path(ruta)
    if not ruta.exists():
        print('No existe, se omite:', ruta)
        return 0
    n = 0
    for item in ruta.rglob('*'):
        if item.is_file():
            destino = Path(carpeta_zip) / item.relative_to(raiz_drive)
            archivo_zip.write(item, destino.as_posix(), compress_type=ZIP_DEFLATED, compresslevel=1)
            n += 1
    return n
raiz = DRIVE_ROOT.parent
zip_path = Path('/content') / f'resultados_B4_{RUN_ID}.zip'
with ZipFile(zip_path, 'w') as zf:
    cantidad = 0
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'b4' / 'ablaciones' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'b4' / 'h2' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'lightgbm' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'b4_reproduccion' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
if cantidad:
    print('Archivos:', cantidad, '| ZIP:', zip_path, '| tamaño MiB:', round(zip_path.stat().st_size / 2**20, 1))
    files.download(str(zip_path))
else:
    zip_path.unlink(missing_ok=True)
    print('No se creó un ZIP: esta corrida todavía no tiene experimentos guardados en Drive.')
